# 01.4 — Classes and OOP

Python is **object-oriented to its core** — every value is an instance of some class, and you'll write your own classes constantly when porting MATLAB code. This is also the foundation for understanding `torch.nn.Module`, the PyTorch base class that every model in `src/neural_data_decoding/models/` inherits from.

This notebook covers:

* Class declarations and `__init__`.
* Instance methods, the `self` parameter.
* `@dataclass` — Python's lightweight struct.
* Inheritance — single and `super().__init__()`.
* `@property`, `@staticmethod`, `@classmethod`.

**Prerequisite:** [01.1 syntax basics](01.1_syntax_basics.ipynb), [01.3 functions](01.3_functions_and_lambdas.ipynb).

## Section 1 — What MATLAB does

MATLAB has `classdef` (introduced in R2008a). The conventions feel verbose but the mechanics are similar to Python.

```matlab
classdef Sweep
    properties
        Name
        Description
        Overrides
    end

    methods
        function obj = Sweep(name, description)
            obj.Name = name;
            obj.Description = description;
            obj.Overrides = struct();
        end

        function out = summarize(obj)
            out = sprintf('Sweep %s: %s', obj.Name, obj.Description);
        end
    end
end
```

Python's equivalent is shorter and reads more naturally.

## Section 2 — The Python concepts you need

### 2.1 — Defining a class

`class` opens a class. Inside, `def __init__(self, ...)` is the constructor. The first parameter of every method is `self` (MATLAB's `obj`).

In [ ]:
class Sweep:
    """A single sweep configuration."""

    def __init__(self, name: str, description: str):
        self.name = name              # instance attributes set in __init__
        self.description = description
        self.overrides: dict = {}

    def summarize(self) -> str:
        return f"Sweep {self.name}: {self.description}"


s = Sweep("Optimal", "Variational + Confidence + MIL")
print(s.summarize())
print(s.name)

**Key things to notice:**

- `self` is the first parameter of every method. You DON'T pass it when calling — Python fills it in automatically. (`s.summarize()` actually calls `Sweep.summarize(s)`.)
- Attributes are set inside methods via `self.attr = value`. There's no separate `properties` block.
- `class Sweep:` (no parentheses) is sufficient for a basic class. The trailing `:` opens an indented body, same as `def` / `if` / `for`.
- Type annotations (`name: str`) are optional but recommended — pyright uses them to catch bugs.

### 2.2 — `@dataclass` — the lightweight Python struct

If your class is just a bag of attributes plus a constructor, the `@dataclass` decorator does the boilerplate for you. This is the **default choice** in our codebase for config-like classes (`SweepEntry`, `MatlabRunDirs`, `RunBannerData`, etc.).

In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True, slots=True)
class SweepEntry:
    """One sweep configuration — flat index + override bundle."""
    sweep_index: int
    matlab_choice: int
    matlab_idx: int
    description: str

# __init__, __repr__, __eq__ are all auto-generated by @dataclass.
entry = SweepEntry(sweep_index=1, matlab_choice=1, matlab_idx=1, description="Feedforward")
print(entry)            # auto __repr__ shows all fields
print(entry.description) # field access via dot, like any class

**Decorator options worth knowing:**

- `frozen=True` — instances are immutable after construction (assignment raises). Use for value-like classes you compare by content.
- `slots=True` — uses Python's `__slots__` mechanism for slight memory + speed wins, and rejects unknown attributes (great for catching typos).
- `kw_only=True` — forces every field to be passed by keyword, never positionally.

The real `SweepEntry` in `src/neural_data_decoding/sweeps/dispatcher.py` uses `frozen=True, slots=True` — go look.

### 2.3 — Inheritance and `super()`

Python supports single inheritance (and limited multiple inheritance). The base class goes in parentheses after the child class name.

In [ ]:
class Animal:
    def __init__(self, name: str):
        self.name = name

    def speak(self) -> str:
        return "..."


class Dog(Animal):
    def __init__(self, name: str, breed: str):
        super().__init__(name)        # call the base class __init__
        self.breed = breed

    def speak(self) -> str:            # override the base method
        return "woof"


d = Dog("Rex", "GSD")
print(d.name, d.breed, d.speak())

Worth a picture. The diagram below shows the inheritance relationship from `Animal` → `Dog` from the example above, and on the right the production case in `src/neural_data_decoding/` — `MatFileTrialDataset` inherits from PyTorch's `Dataset` base class in `torch.utils.data` (a separate hierarchy from `nn.Module`; datasets describe data access, modules describe computation).

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

def draw_box(ax, x, y, w, h, label, color="#cce4ff"):
    rect = mpatches.FancyBboxPatch((x - w / 2, y - h / 2), w, h,
        boxstyle="round,pad=0.05", facecolor=color, edgecolor="black", linewidth=1.5)
    ax.add_patch(rect)
    if "\n" in label:
        head, tail = label.split("\n", 1)
        ax.text(x, y + 0.08, head, ha="center", va="center", fontsize=12, fontweight="bold")
        ax.text(x, y - 0.18, tail, ha="center", va="center", fontsize=10, family="monospace")
    else:
        ax.text(x, y, label, ha="center", va="center", fontsize=12, fontweight="bold")

def arrow(ax, x1, y1, x2, y2, label=""):
    ax.annotate("", xy=(x2, y2), xytext=(x1, y1),
        arrowprops=dict(arrowstyle="->", color="black", lw=1.5))
    if label:
        ax.text((x1 + x2) / 2 + 0.2, (y1 + y2) / 2, label, fontsize=10, color="#666", style="italic")

# Left: Animal / Dog example
ax1.set_title("Teaching example", fontsize=12)
draw_box(ax1, 2, 4, 2.6, 0.9, "Animal\nname, speak()", color="#cce4ff")
draw_box(ax1, 2, 1.2, 2.6, 0.9, "Dog(Animal)\nname, breed, speak()", color="#ffe4cc")
arrow(ax1, 2, 3.4, 2, 1.8, "inherits")
ax1.text(2, 0.2, "Dog reuses name from Animal,\nadds breed, overrides speak()",
    ha="center", fontsize=9, color="#666")
ax1.set_xlim(0, 4); ax1.set_ylim(-0.2, 5); ax1.set_aspect("equal"); ax1.axis("off")

# Right: Production inheritance
ax2.set_title("Production case in this codebase", fontsize=12)
draw_box(ax2, 2, 4.2, 3.2, 0.9, "Dataset\ntorch.utils.data", color="#dddddd")
draw_box(ax2, 2, 2.2, 3.2, 0.9, "MatFileTrialDataset\nour data/mat_dataset.py", color="#ffe4cc")
draw_box(ax2, 2, 0.2, 3.2, 0.9, "(used by DataLoader)\nin training loop", color="#eaf7e6")
arrow(ax2, 2, 3.6, 2, 2.8, "inherits")
arrow(ax2, 2, 1.6, 2, 0.85)
ax2.text(2, -0.7, "__len__ + __getitem__ contract\nis what DataLoader expects",
    ha="center", fontsize=9, color="#666", style="italic")
ax2.set_xlim(0, 4); ax2.set_ylim(-1, 5); ax2.set_aspect("equal"); ax2.axis("off")

plt.tight_layout()
plt.show()
print("Inheritance is the foundation of nn.Module — covered next in Module 02.6.")

**`super().__init__()` is the rule, not the exception.** Forgetting it leaves base-class attributes uninitialized — a common source of bugs.

PyTorch modules ALWAYS call `super().__init__()` first thing in their `__init__`:

```python
class MyModel(nn.Module):
    def __init__(self):
        super().__init__()        # CRITICAL — initializes nn.Module bookkeeping
        self.fc = nn.Linear(10, 5)
```

If you omit the `super()` call, PyTorch fails fast: the moment you assign a submodule (`self.fc = nn.Linear(...)`) it raises `AttributeError: cannot assign module before Module.__init__() call`. A loud, immediate error — much kinder than a silent one.

### 2.4 — Special methods (dunder methods)

Python uses `__double_underscore__` ("dunder") names for magic methods that hook into language operators. The big ones:

| Method | Triggers when | Example |
|---|---|---|
| `__init__` | constructor | `MyClass(x)` |
| `__repr__` | `repr(obj)` or bare echo | what a notebook cell shows when the object is the last expression |
| `__str__` | `str(obj)` or `print(obj)` | string representation |
| `__eq__` | `a == b` | equality |
| `__len__` | `len(obj)` | counting |
| `__getitem__` | `obj[i]` | indexing |
| `__call__` | `obj(...)` | making the instance callable |

Datasets implement `__len__` and `__getitem__`. PyTorch modules implement `__call__` (which is what makes `model(x)` work).

In [ ]:
class Counter:
    def __init__(self, items: list):
        self.items = items

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        return self.items[idx] * 2


c = Counter([10, 20, 30])
print(len(c))      # 3
print(c[0])        # 20 — __getitem__ doubles it
for x in c:
    print(x)

### 2.5 — Properties — methods that look like attributes

`@property` turns a method into a read-only attribute. Used when you want a computed value to feel like a field.

In [ ]:
class Trial:
    def __init__(self, raw_features):
        self.raw_features = raw_features

    @property
    def num_samples(self):
        """Number of samples in this trial."""
        return len(self.raw_features)


t = Trial([1, 2, 3, 4, 5])
print(t.num_samples)        # called without parentheses!

You'll see `@property` in `MatFileTrialDataset.session_ids`, `.num_dimensions`, `.trial_ids` — every dataset class exposes derived state this way.

### 2.6 — `@staticmethod` and `@classmethod`

`@staticmethod` — a function that lives in the class namespace but doesn't take `self`. Use it for utilities that are conceptually related to the class.

`@classmethod` — takes the class itself as the first parameter (conventionally `cls`). Used for alternative constructors.

In [ ]:
class TrialID:
    def __init__(self, value: int):
        self.value = value

    @classmethod
    def from_filename(cls, filename: str):
        """Alternative constructor — parses an ID from a filename."""
        digits = filename.split("_")[-1].split(".")[0]
        return cls(int(digits))         # `cls` is TrialID

    @staticmethod
    def is_valid_filename(filename: str) -> bool:
        """Utility predicate that doesn't need an instance."""
        return filename.startswith("Decision_Data_")


tid = TrialID.from_filename("Decision_Data_0000011.mat")
print(tid.value)
print(TrialID.is_valid_filename("Decision_Data_0000011.mat"))

## Section 3 — The `neural_data_decoding` implementation

Look at how `MatFileTrialDataset` uses every concept in this notebook.

In [ ]:
from pathlib import Path
src = Path("../../src/neural_data_decoding/data/mat_dataset.py").read_text().splitlines()
# Find the class definition
class_start = next(
    (i for i, line in enumerate(src) if line.startswith("class MatFileTrialDataset")),
    -1,
)
for i in range(class_start, min(class_start + 40, len(src))):
    print(f"{i + 1:3} | {src[i]}")

Things to spot:

- `class MatFileTrialDataset(Dataset):` — inherits from PyTorch's `Dataset`.
- `def __init__(self, *, data_dir, target_dir, ...)`: the `*` makes EVERY argument keyword-only (no positional calls allowed).
- No explicit `super().__init__()` call — `torch.utils.data.Dataset` defines no `__init__` of its own, so there is nothing to initialize. (`nn.Module` is the opposite case: its `__init__` is mandatory.)
- `@property def session_ids(self):` — derived attribute used by samplers.
- `def __len__(self):` and `def __getitem__(self, idx):` — the two methods PyTorch's DataLoader expects.

## Section 4 — Hands-on exercises

### Exercise 4.1 — port a MATLAB classdef

Port this:

```matlab
classdef Point
    properties
        x
        y
    end
    methods
        function obj = Point(x, y)
            obj.x = x;
            obj.y = y;
        end
        function d = distance(obj, other)
            d = sqrt((obj.x - other.x)^2 + (obj.y - other.y)^2);
        end
    end
end
```

In [ ]:
# Your attempt here


In [ ]:
# Reveal:
from math import sqrt

class Point:
    def __init__(self, x: float, y: float):
        self.x = x
        self.y = y

    def distance(self, other: "Point") -> float:
        return sqrt((self.x - other.x) ** 2 + (self.y - other.y) ** 2)


p1 = Point(0, 0)
p2 = Point(3, 4)
print(p1.distance(p2))    # 5.0

### Exercise 4.2 — use `@dataclass` instead

Rewrite Exercise 4.1 using `@dataclass(frozen=True)` so the boilerplate is gone.

In [ ]:
# Your attempt here


In [ ]:
# Reveal:
from dataclasses import dataclass
from math import sqrt

@dataclass(frozen=True, slots=True)
class Point2:
    x: float
    y: float

    def distance(self, other: "Point2") -> float:
        return sqrt((self.x - other.x) ** 2 + (self.y - other.y) ** 2)


p1 = Point2(0, 0)
p2 = Point2(3, 4)
print(p1.distance(p2))
print(p1)    # auto __repr__

## Section 5 — Common errors

### `TypeError: Sweep.summarize() missing 1 required positional argument: 'self'`

Almost always means you called the method on the class instead of an instance: `Sweep.summarize()` instead of `s.summarize()`. The class-level call has no instance to fill in `self`, so Python reports the method itself as missing an argument.

### `AttributeError: 'Sweep' object has no attribute 'x'`

You tried to read `s.x` but `x` was never set in `__init__` (or you misspelled it). With `@dataclass(slots=True)` this is caught at assignment time, not read time — `slots` rejects unknown attributes.

### Forgetting `super().__init__()`

```python
class MyModel(nn.Module):
    def __init__(self):
        # super().__init__() missing!
        self.fc = nn.Linear(10, 5)   # raises AttributeError right here
```

Constructing `MyModel()` raises `AttributeError: cannot assign module before Module.__init__() call` — PyTorch guards the assignment of submodules so this mistake can't slip through silently.

### Mutable class-level defaults

```python
class Sweep:
    overrides = {}     # WRONG — shared across all instances

    def __init__(self, name):
        self.name = name
```

Every `Sweep()` instance shares the same `overrides` dict, just like the mutable-default-argument bug. Use `__init__` to set per-instance state, or `@dataclass(field(default_factory=dict))`.

## Section 6 — Further reading

- [Python tutorial — classes](https://docs.python.org/3/tutorial/classes.html). The official walkthrough.
- [dataclasses docs](https://docs.python.org/3/library/dataclasses.html). Every option for `@dataclass`.
- [PyTorch nn.Module docs](https://pytorch.org/docs/stable/generated/torch.nn.Module.html). The base class for every model in this codebase.
- [PEP 557 — dataclasses](https://peps.python.org/pep-0557/). The original proposal; explains the design.

Module 01 continues with **[01.5 modules and imports](01.5_modules_and_imports.ipynb)**, **[01.6 error handling](01.6_error_handling.ipynb)**, **[01.7 dataclasses and typed configs](01.7_dataclasses_and_typed_configs.ipynb)**, and **[01.8 the Python standard library](01.8_the_python_standard_library_for_matlab_users.ipynb)** (next curriculum batch).